# Lab 02: Reading & Writing Data Formats

## Objectives
- Read data from CSV, JSON, and Parquet formats
- Understand schema inference vs explicit schema definition
- Write data as Delta tables for downstream labs
- Compare format characteristics (file size, schema support, read speed)

## Exam Domain
- **Developing DataFrame/DataSet API Applications — 30%**

## Estimates
- **Time:** ~30 minutes
- **Cost:** ~$1 (serverless)
- **Compute:** Serverless

![Data Formats I/O](../assets/diagrams/lab-02-data-formats-io.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

TAXI_PATH = "/Volumes/spark_lab_guide/default/nyc_taxi"
GITHUB_PATH = "/Volumes/spark_lab_guide/default/github_archive"

## A. Reading Parquet Files

Parquet is a columnar binary format with embedded schema. It is the most efficient format for Spark because it supports predicate pushdown and column pruning — Spark only reads the columns and rows it needs.

In [ ]:
# Read Parquet — schema is embedded, no options needed
taxi_df = spark.read.parquet(TAXI_PATH)

print(f"Rows: {taxi_df.count():,}")
print(f"Columns: {len(taxi_df.columns)}")
taxi_df.printSchema()

In [ ]:
# Preview the data
taxi_df.show(5, truncate=False)

## B. Reading JSON Files

JSON is human-readable but slow for Spark: text-based, row-oriented, and schema must be inferred by scanning the data. The GitHub Archive data is newline-delimited JSON (one JSON object per line).

In [ ]:
# Read JSON with schema inference (Spark scans the data to determine types)
github_df = spark.read.json(GITHUB_PATH)

print(f"Rows: {github_df.count():,}")
print(f"Columns: {len(github_df.columns)}")
github_df.printSchema()

In [ ]:
# Preview — note the nested structure
github_df.select("id", "type", "actor.login", "repo.name", "created_at").show(5, truncate=False)

> **Exam Tip:** The exam tests when to use schema inference vs explicit schema. Key rule: **always define schema explicitly in production** — inference is slow (requires an extra pass over the data) and can guess wrong types. Inference is fine for exploration.

## C. Defining Schema Explicitly

Instead of letting Spark infer the schema, you can define it using `StructType`. This is faster and more reliable.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType, IntegerType

# Explicit schema for taxi data (subset of columns for clarity)
taxi_schema = StructType([
    StructField("VendorID", LongType()),
    StructField("tpep_pickup_datetime", TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count", DoubleType()),
    StructField("trip_distance", DoubleType()),
    StructField("PULocationID", LongType()),
    StructField("DOLocationID", LongType()),
    StructField("payment_type", LongType()),
    StructField("fare_amount", DoubleType()),
    StructField("tip_amount", DoubleType()),
    StructField("total_amount", DoubleType()),
])

# Read with explicit schema — no inference pass needed
taxi_explicit = spark.read.schema(taxi_schema).parquet(TAXI_PATH)
taxi_explicit.printSchema()

## D. Reading CSV (for comparison)

CSV requires the most configuration: you must specify whether the file has headers, the delimiter, and how to handle types. Let's simulate by writing taxi data as CSV and reading it back.

In [ ]:
# Write a small sample as CSV to demonstrate CSV reading
sample = taxi_df.limit(1000)
csv_path = "/Volumes/spark_lab_guide/default/github_archive/taxi_sample_csv"
sample.write.mode("overwrite").option("header", True).csv(csv_path)

# Read it back — note the options needed
csv_df = (
    spark.read
    .option("header", True)        # first row is column names
    .option("inferSchema", True)   # infer types (slow but convenient)
    .csv(csv_path)
)
csv_df.printSchema()
csv_df.show(3)

### SQL Alternative

You can also read files directly using SQL with the `read_files` table-valued function.

In [ ]:
# SQL: reading files directly
spark.sql(f"""
    SELECT * FROM read_files(
        '{TAXI_PATH}',
        format => 'parquet'
    )
    LIMIT 5
""").show(truncate=False)

## E. Writing Delta Tables

Delta is the recommended format for Databricks. It adds ACID transactions, time travel, and schema enforcement on top of Parquet. We'll write both datasets as Delta tables — these tables are used by all remaining labs.

In [ ]:
# Write taxi data as a managed Delta table
(
    taxi_df
    .write
    .mode("overwrite")
    .saveAsTable("taxi_trips")
)

print("taxi_trips table created.")
spark.sql("SELECT count(*) as row_count FROM taxi_trips").show()

In [ ]:
# Write GitHub data as a managed Delta table
(
    github_df
    .write
    .mode("overwrite")
    .saveAsTable("github_events")
)

print("github_events table created.")
spark.sql("SELECT count(*) as row_count FROM github_events").show()

## F. Write Modes

Spark supports four write modes that control behavior when the target already exists:

| Mode | Behavior |
|------|----------|
| `error` (default) | Throws an error if target exists |
| `overwrite` | Replaces existing data |
| `append` | Adds rows to existing data |
| `ignore` | Silently does nothing if target exists |

In [ ]:
# Demonstrate append mode — add more data to an existing table
small_sample = taxi_df.limit(100)

small_sample.write.mode("append").saveAsTable("taxi_trips")
print("Appended 100 rows to taxi_trips")

# Check new count
spark.sql("SELECT count(*) as row_count FROM taxi_trips").show()

In [ ]:
# Restore original data (overwrite back)
taxi_df.write.mode("overwrite").saveAsTable("taxi_trips")
print("Restored taxi_trips to original data")
spark.sql("SELECT count(*) as row_count FROM taxi_trips").show()

> **Exam Tip:** Know all four write modes and their behavior. Common trap: the default mode is `error` (also called `errorifexists`), not `overwrite`. If you forget to set the mode and the table exists, your write will fail.

## Exam-Style Review

**Q1.** What happens when you call `spark.read.json(path)` without providing a schema?
- A) Spark throws an error
- B) Spark infers the schema by scanning the data (extra read pass)
- C) All columns are read as StringType
- D) Spark uses a default schema

**Q2.** Which file format embeds its schema and supports column pruning natively?
- A) CSV
- B) JSON
- C) Parquet
- D) Text

**Q3.** What is the default write mode in Spark?
- A) overwrite
- B) append
- C) ignore
- D) error

**Q4.** What advantage does Delta format have over plain Parquet?
- A) Faster read speed
- B) ACID transactions, time travel, and schema enforcement
- C) Smaller file size
- D) No schema required

### Answers
- **Q1: B** — Spark scans the data to infer types, requiring an extra pass.
- **Q2: C** — Parquet is columnar with embedded schema, supporting column pruning and predicate pushdown.
- **Q3: D** — The default is `error` (or `errorifexists`). If the target exists, the write fails.
- **Q4: B** — Delta adds ACID transactions, time travel, and schema enforcement on top of Parquet.

## Key Takeaways
- Parquet is the most efficient read format (columnar, schema embedded, supports pushdown)
- JSON requires schema inference or explicit schema; it is slower than Parquet
- Always define schema explicitly in production pipelines
- Delta is the recommended write format — it adds ACID, time travel, and schema enforcement
- Know all four write modes: `error` (default), `overwrite`, `append`, `ignore`

**Tables created in this lab:** `taxi_trips`, `github_events` — used by all remaining labs.

**Next:** [Lab 03 — DataFrame Transformations & Actions](03-dataframe-transformations-actions.ipynb)

In [ ]:
# Cleanup — remove temporary CSV files only
# DO NOT drop taxi_trips or github_events — they are used by subsequent labs
try:
    dbutils.fs.rm("/Volumes/spark_lab_guide/default/github_archive/taxi_sample_csv", True)
except Exception:
    pass
print("Lab 02 cleanup complete. taxi_trips and github_events tables preserved.")